In [1]:
# using pretrained VIT 
import sys
# !{sys.executable} -m pip install Transformers

In [2]:
import torch
import torch.nn as nn
from transformers import ViTModel, ViTConfig


In [3]:
# Config
NUM_CLASSES = 20
NUM_FRAMES = 16
IMG_SIZE = 224

In [ ]:
from transformers import VideoMAEModel

class ASLVideoMAE(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.videomae = VideoMAEModel.from_pretrained("MCG-NJU/videomae-base")
        hidden_size = self.videomae.config.hidden_size  # 768
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        # x shape: (batch, num_frames, C, H, W)
        # VideoMAE expects (batch, C, T, H, W) — permute
        
        outputs = self.videomae(pixel_values=x)
        # mean pool over the sequence of patch tokens
        features = outputs.last_hidden_state.mean(dim=1)  # (B, 768)
        return self.classifier(features)

In [ ]:
# Updated test block at the bottom of models.ipynb
model = ASLVideoMAE(num_classes=NUM_CLASSES)
dummy_input = torch.randn(2, NUM_FRAMES, 3, IMG_SIZE, IMG_SIZE)
output = model(dummy_input)

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Input shape: torch.Size([2, 16, 3, 224, 224])
Output shape: torch.Size([2, 20])
Expected output: (2, 20)
